In [1]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from neo4j import GraphDatabase
import os
from hashlib import md5
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import tqdm
from tqdm import tqdm
import json
from random import sample
import threading
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
from typing import Dict, Any, List
import asyncio
import json
from typing import Any, Dict, List, Optional, Sequence, Tuple, Type, Union, cast
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document
from langchain_core.language_models import BaseLanguageModel
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    PromptTemplate,
)
from changing_names import load_name_mapping, save_name_mapping, find_names, substitute_names
from langchain_core.pydantic_v1 import BaseModel, Field, create_model

[nltk_data] Downloading package punkt to /u/sebono/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package names to /u/sebono/nltk_data...
[nltk_data]   Package names is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /u/sebono/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [2]:
from langchain.graphs import Neo4jGraph
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from hashlib import md5
from typing import Dict, Any, List

BASE_ENTITY_LABEL = "__Entity__"
EXCLUDED_LABELS = ["_Bloom_Perspective_", "_Bloom_Scene_"]
EXCLUDED_RELS = ["_Bloom_HAS_SCENE_"]
INCLUDE_DOCS_QUERY = (
    "MERGE (d:Document {id:$document.metadata.id}) "
    "SET d.text = $document.page_content "
    "SET d += $document.metadata "
    "WITH d "
)

from langchain_core.load.serializable import Serializable

class GraphManager(Neo4jGraph):
    def __init__(self, uri: str, user: str, password: str, first_node: str, relationship_property_descriptions: Dict[Any,Any]):
        """
        Initialize the GraphManager, extending Neo4jGraph.
        """
        super().__init__(url=uri, username=user, password=password)
        self.uri = uri
        self.user = user
        self.password = password
        self.char_name = first_node
        self.relationship_properties = list(relationship_property_descriptions.keys())
        self.relationship_property_descriptions = relationship_property_descriptions
        self.second_layer_nodes = []
        self.subgraphs_schemas = {}
        self.second_layer_nodes_schemas = {}
        self.compute_second_layer_nodes()
        self.compute_subgraphs()
    
    def reconnect(self):
        """
        Refresh the connection and reinitialize the GraphManager.
        """
        super().__init__(url=self.uri, username=self.user, password=self.password)
    
    def update_relationship_properties(self, clusters_with_explanations):
        self.relationship_properties+=list(clusters_with_explanations.keys())
        # Merge dictionaries, prioritizing the new clusters
        self.relationship_property_descriptions = {
            **self.relationship_property_descriptions,
            **clusters_with_explanations,
        }
    
    # Connection Reconfiguration Methods
    def update_graph_relationships(self, rels: List[Dict[str, Any]]):
        """
        Updates relationships in the graph database with enriched properties.

        Args:
            rels (List[Dict[str, Any]]): List of enriched relationships with properties.

        Returns:
            None
        """
        print(f"Updating {len(rels)} relationships with new Clusters...")
        for rel in tqdm(rels):
            # Dynamically create the query string with the relationship type
            query = f"""
            MATCH (source)-[r:{rel['type']}]->(target)
            WHERE source.id = $source_node_id AND target.id = $target_node_id
            SET r += {"{" + ", ".join(
                    [f"{key}: {value}" for key,value in rel["properties"].items()]
                )+"}"}
            """
            query1 = f"""
            MATCH (source)-[r:MENTIONS]-(doc:Document)
            MATCH (doc)-[:MENTIONS]-(target)
            MATCH (source)-[rel]->(target)
            WHERE source.id = $source_node_id AND target.id = $target_node_id
            SET r += {"{" + ", ".join(
                    [f"{key}: {value}" for key,value in rel["properties"].items()]
                )+"}"}
            """
            # Prepare the parameters for the query
            params = {
                "source_node_id": rel["source_node_id"],
                "target_node_id": rel["target_node_id"],
            }
            # Print the query and parameters for debugging
            substituted_query = query.replace("$source_node_id", f"'{params['source_node_id']}'") \
                              .replace("$target_node_id", f"'{params['target_node_id']}'")
            # Execute the query
            self.query(query, params)
            self.query(query1, params)


    def reconfigure_connections(self, node_type: str, attribute_name: str, subgraph: str):
        """
        Reconfigure graph connections for a specific node type and attribute.
        """
        query = f"""
            MATCH (source {{id: '{subgraph}'}})-[r]-(target)
            WHERE ANY(prop IN keys(r) WHERE prop = '{node_type}' AND '{attribute_name}' IN r[prop])
            WITH source, r, target, type(r) AS relType, properties(r) AS props

            // Create or find the intermediate node
            MERGE (intermediate:{attribute_name.capitalize()} {{id: '{attribute_name.capitalize()}'}})
            SET intermediate.type = 'Intermediate_{node_type}', intermediate.subgraphid = 'SecLayerNode', intermediate.description="{self.relationship_property_descriptions[attribute_name]}"
            SET source.subgraphid = 'SecLayerNode'

            // Connect the intermediate node to the subgraph
            WITH source, target, intermediate, props, relType, r
            MATCH (subgraph {{id: '{subgraph}'}})
            WHERE intermediate <> subgraph
            MERGE (subgraph)-[:HAS_SUBGRAPH]-(intermediate)
              ON CREATE SET subgraph.subgraphid = 'SecLayerNode'

            // Recreate the relationship from the intermediate node to the target
            WITH intermediate, target, props, relType, r
            CALL apoc.create.relationship(intermediate, relType, {{sentence: r.sentence}}, target) YIELD rel AS newRel

            // Remove `{attribute_name}` from the `{node_type}` property
            WITH r, newRel
            SET r.{node_type} = [attr IN r.{node_type} WHERE attr <> '{attribute_name}']

            // Remove the `{node_type}` property if it becomes empty
            WITH r, newRel
            FOREACH (_ IN CASE WHEN r.{node_type} IS NOT NULL AND size(r.{node_type}) = 0 THEN [1] ELSE [] END |
                REMOVE r.{node_type}
            )

            // Remove the relationship if all its properties are empty while ignoring the sentence field
            // WITH r
            // WHERE ALL(prop IN keys(r) 
            //           WHERE prop = "sentence" OR r[prop] IS NULL OR (r[prop] IS LIST AND size(r[prop]) = 0))
            // DELETE r
        """
        print(query)
        self.query(query)
    
    def set_subgraph_status_on(self, intermediate, depth: Optional[int] = 2):
        query = f"""
        MATCH path = (start {{id: '{intermediate}'}})-[relationships*1..{depth}]->(nodes)
        WHERE NOT nodes:Document 
          AND (nodes.type IS NULL OR NOT nodes.type STARTS WITH 'Intermediate_')
        WITH nodes
        //SET nodes.status = 'on'
        RETURN COUNT(nodes) AS UpdatedNodesCount
        """
        self.query(query)
        
    def reconfigure_all_connections(self, data: Dict[str, List[str]], subgraph: str):
        """
        Reconfigure all connections based on a dictionary of node types and attributes.
        """
        for node_type, attributes in data.items():
            for attribute in attributes:
                self.reconfigure_connections(node_type, attribute, subgraph)

    # Node Management Methods
    def merge_duplicate_nodes(self):
        """
        Merge nodes with duplicate IDs into a single node.
        """
        query = """
        MATCH (n)
        WITH n.id AS nodeId, COLLECT(n) AS nodes
        WHERE SIZE(nodes) > 1
        CALL apoc.refactor.mergeNodes(nodes) YIELD node
        RETURN node
        """
        self.query(query)

    def delete_nodes(self, node_ids: List[str]):
        """
        Delete a list of nodes by their 'id' property.
        
        Parameters:
        - node_ids (List[str]): List of node IDs to be deleted.
        """
        query = """
        UNWIND $node_ids AS node_id
        MATCH (n {id: node_id})
        DETACH DELETE n
        """
        self.query(query, {"node_ids": node_ids})
    
    def delete_relationships_not_connecting_2nd_layer_nodes(self, node_ids: List[str]):
        """
        Delete relationships for a list of nodes where the target node's 'type' property
        does not start with 'Intermediate_'.

        Parameters:
        - node_ids (List[str]): List of node IDs whose relationships should be deleted.
        """
        query = """
        UNWIND $node_ids AS node_id
        MATCH (n {id: node_id})-[r]-(target)
        WHERE (target.type IS NULL OR NOT target.type STARTS WITH 'Intermediate_') AND NOT target:Document
        DELETE r
        """
        self.query(query, {"node_ids": node_ids})

    # Relationship Management Methods
    def merge_duplicate_relationships(self):
        """
        Merge all duplicate relationships based on unique properties.
        This function combines all relationships of the same type and properties between the same nodes.
        """
        query = """
        MATCH (n)-[r]->(m)
        WITH n, m, type(r) AS relType, COLLECT(r) AS rels
        WHERE SIZE(rels) > 1
        CALL apoc.refactor.mergeRelationships(rels) YIELD rel
        RETURN rel
        """
        self.query(query)

    # Document Import Methods
    def add_graph_documents(
        self,
        graph_documents: List[GraphDocument],
        subgraphs: Optional[Dict[Any,Any]]=None,
        include_source: bool = False,
        baseEntityLabel: bool = False,
        
    ) -> None:
        """
        This method constructs nodes and relationships in the graph based on the
        provided GraphDocument objects.

        Parameters:
        - graph_documents (List[GraphDocument]): A list of GraphDocument objects
        that contain the nodes and relationships to be added to the graph. Each
        GraphDocument should encapsulate the structure of part of the graph,
        including nodes, relationships, and the source document information.
        - include_source (bool, optional): If True, stores the source document
        and links it to nodes in the graph using the MENTIONS relationship.
        This is useful for tracing back the origin of data. Merges source
        documents based on the `id` property from the source document metadata
        if available; otherwise it calculates the MD5 hash of `page_content`
        for merging process. Defaults to False.
        - baseEntityLabel (bool, optional): If True, each newly created node
        gets a secondary __Entity__ label, which is indexed and improves import
        speed and performance. Defaults to False.
        """
        if baseEntityLabel:  # Check if constraint already exists
            constraint_exists = any(
                [
                    el["labelsOrTypes"] == [BASE_ENTITY_LABEL]
                    and el["properties"] == ["id"]
                    for el in self.structured_schema.get("metadata", {}).get(
                        "constraint", []
                    )
                ]
            )

            if not constraint_exists:
                # Create constraint
                self.query(
                    f"CREATE CONSTRAINT IF NOT EXISTS FOR (b:{BASE_ENTITY_LABEL}) "
                    "REQUIRE b.id IS UNIQUE;"
                )
                self.refresh_schema()  # Refresh constraint information

        node_import_query = self._get_node_import_query(baseEntityLabel, include_source)
        rel_import_query = self._get_rel_import_query(baseEntityLabel)
        for document in graph_documents:
            if not document.source.metadata.get("id"):
                document.source.metadata["id"] = md5(
                    document.source.page_content.encode("utf-8")
                ).hexdigest()

            # Remove backticks from node types and ensure non-empty types
            for node in document.nodes:
                node.type = self._remove_backticks(node.type)
                if not node.type:
                    raise ValueError(f"Node type cannot be empty: {node}")

            # Import nodes
            self.query(
                node_import_query,
                {
                    "data": [el.__dict__ for el in document.nodes],
                    "document": document.source.__dict__,
                },
            )
            # Import relationships
            for el in document.relationships:
                if subgraphs:
                    for subgraph in subgraphs.keys():
                        if subgraph not in el.properties:
                            el.properties[subgraph] = []
                        # Use a set to ensure uniqueness
                        el.properties[subgraph] = list(set(el.properties[subgraph] + list(subgraphs[subgraph].keys())))
                self.query(
                    rel_import_query,
                    {
                        "data": [
                            {
                                "source": el.source.id,
                                "source_label": self._remove_backticks(el.source.type),
                                "target": el.target.id,
                                "target_label": self._remove_backticks(el.target.type),
                                "type": self._remove_backticks(
                                    el.type.replace(" ", "_").upper()
                                ),
                                "properties": el.properties,
                            }
                        ]
                    },
                )
                if include_source:
                    # Add MENTIONS relationships
                    mentions_properties = self._dict_to_cypher_map(el.properties)
                    self.query(
                        f"""
                        MATCH (d:Document {{id: $document_id}})
                        MATCH (n {{id: $node_id}})
                        MERGE (d)-[:MENTIONS  {mentions_properties}]->(n)
                        """,
                        {
                            "document_id": document.source.metadata["id"],
                            "node_id": el.source.id,
                        },
                    )
                    self.query(
                        f"""
                        MATCH (d:Document {{id: $document_id}})
                        MATCH (n {{id: $node_id}})
                        MERGE (d)-[:MENTIONS]->(n)
                        """,
                        {
                            "document_id": document.source.metadata["id"],
                            "node_id": el.target.id,
                        },
                    )
    
    def get_schema(self, query):
        results = self.query(query)

        node_props = {}
        rel_props = {}
        relationships = []

        for record in results:
            start_types = record["StartNodeType"]  # Now handling multiple possible labels
            node_types = record["NodeTypes"]

            for start_type in start_types:
                if start_type not in node_props:
                    node_props[start_type] = []
                for k in record["NodeProperties"].keys():
                    if k is not None and all(prop["property"] != k for prop in node_props[start_type]):
                        node_props[start_type].append({"property": k, "type": "STRING"})

            for node_type in node_types:
                if node_type not in node_props:
                    node_props[node_type] = []
                for k in record["NodeProperties"].keys():
                    if k is not None and all(prop["property"] != k for prop in node_props[node_type]):
                        node_props[node_type].append({"property": k, "type": "STRING"})

            rel_type = record["RelationshipType"]
            if rel_type not in rel_props and record["RelationshipProperties"].keys():
                rel_props[rel_type] = []
            for k in record["RelationshipProperties"].keys():
                if k is not None and all(prop["property"] != k for prop in rel_props[rel_type]):
                    rel_props[rel_type].append({"property": k, "type": "STRING"})

            relationships.append({
                "start": start_types[0],  # Typically, the primary type if multiple #took out [0]
                "type": rel_type,
                "end": node_types[0]
            })

        return {
            "node_props": node_props,
            "rel_props": rel_props,
            "relationships": relationships
        }
    
    # Schema and Cypher Chain Methods
    def compute_subgraphs(self, depth: Optional[int] = 1):
        self.second_layer_nodes = self.get_nodes_by_label("Intermediate_")
        
        for node in tqdm(self.second_layer_nodes):
            query = f"""
           MATCH path = (start {{id: '{node}'}})-[relationships*1..{depth}]->(nodes)
            WHERE NOT nodes:Document 
              //AND nodes.status <> 'off'
              //AND (nodes.type IS NULL OR NOT nodes.type STARTS WITH 'Intermediate_')
            WITH nodes, relationships, start
            UNWIND nodes AS node
            UNWIND relationships AS rel
            RETURN 
                labels(start) AS StartNodeType, start.id AS StartNodeName,
                labels(node) AS NodeTypes, properties(node) AS NodeProperties,
                type(rel) AS RelationshipType, properties(rel) AS RelationshipProperties
            """
            result = self.get_schema(query)
            if result["node_props"] != {}:
                if node not in self.subgraphs_schemas:
                    self.subgraphs_schemas[node] = {}
                self.subgraphs_schemas[node] = result
            else:
                print(f"skipping {node}... no nodes for this category")
    
    def compute_second_layer_nodes(self, depth: Optional[int] = 3):
        
        query = f"""
            MATCH p = (sourceNode {{id: '{self.char_name}', subgraphid: 'SecLayerNode'}})-[:HAS_SUBGRAPH*1..{depth}]->(targetNode)
            WHERE targetNode.subgraphid = 'SecLayerNode'
            WITH nodes(p) AS pathNodes, relationships(p) AS pathRels
            UNWIND range(0, size(pathRels) - 1) AS idx
            WITH 
                DISTINCT pathNodes[idx] AS startNode, 
                pathNodes[idx + 1] AS endNode, 
                pathRels[idx] AS rel
            RETURN 
                labels(startNode) AS StartNodeType, 
                properties(startNode) AS StartProperties, 
                type(rel) AS RelationshipType, 
                properties(rel) AS RelationshipProperties, 
                labels(endNode) AS NodeTypes, 
                properties(endNode) AS NodeProperties
        """
        result = self.get_schema(query)
        if result["node_props"] != {}:
            self.second_layer_nodes_schemas = result
        else:
            print(f"skipping {self.char_name}... no nodes for this category")
    
    def get_nodes_by_label(self, node_type: str) -> Dict[str, str]:
        """
        Return all nodes that have a specific type or a specific id with their id, description, and type.
        
        Args:
        node_type (Optional[str]): The type of the nodes to query. (e.g., "Intermediate_")
        node_id (Optional[str]): The id of a specific node to query.
        
        Returns:
        dict: A dictionary with node id as keys and node info as values.
        """

        query = f"""
        MATCH (n)
        WHERE n.type STARTS WITH '{node_type}'
        RETURN n.id AS id, n.description AS description, labels(n) AS type
        """

        nodes = {}
        result = self.query(query)
        for record in result:
            node_info = {}
            node_info['Node type'] = record['type'][0]
            node_info['id']= record['id']
            node_info['description']= record['description']
            nodes[record['id']] = node_info
        return nodes

    def get_nodes_by_id(self, node_id: str):
        """
        Args:
        node_type (Optional[str]): The type of the nodes to query. (e.g., "Intermediate_")
        node_id (Optional[str]): The id of a specific node to query.
        
        Returns:
        dict: A dictionary with node id as keys and node info as values.
        """
        query = f"""
        MATCH (n)
        WHERE n.id = '{node_id}'
        RETURN n.id AS id, n.description AS description, labels(n) AS type
        """
        
        nodes = {}
        result = self.query(query)
        for record in result:
            node_info = {}
            node_info['Node type'] = record['type'][0]
            node_info['id']= record['id']
            node_info['description']= record['description']
            nodes[record['id']] = node_info
        return nodes
    
    def get_documents_subgraph(self, node_id: str):
        """
        Retrieves all nodes of type Document and their relationships connected to the specified node.

        Args:
            node_id (str): The id of the specific node to query.

        Returns:
            tuple: A tuple containing:
                - dict: A dictionary with document node ids as keys and their information as values.
                - list: A list of relationship details for precise identification.
        """
        query = f"""
        MATCH (n)-[:MENTIONS]->(doc:Document)
        MATCH (doc)-[:MENTIONS]->(m)
        MATCH (n)-[r]->(m)
        WHERE n.id = '{node_id}'
        RETURN doc.id AS id, type(r) AS rel_type, doc.text AS text,
               startNode(r).id AS source_node_id, labels(startNode(r))[0] AS source_node_type,
               endNode(r).id AS target_node_id, labels(endNode(r))[0] AS target_node_type,
               r.sentence AS sentence, r AS rel
        """
        documents = {}
        relationships = []

        # Execute the query
        result = self.query(query)

        # Process results
        for record in result:
            # Collect document information
            document_info = {
                'id': record['id'],
                'sentence': record.get('sentence', ""),
                'text': record.get('text', ""),
            }
            documents[record['id']] = document_info

            # Extract relationship details
            relationship_info = {
                "source_node_id": record.get("source_node_id"),
                "source_node_type": record.get("source_node_type"),
                "target_node_id": record.get("target_node_id"),
                "target_node_type": record.get("target_node_type"),
                "type": record.get("rel_type"),
                "sentence": record.get("sentence"),
                "properties": dict(record["rel"]) if isinstance(record["rel"], dict) else {},
            }
            relationships.append(relationship_info)

        return documents, relationships


    @staticmethod
    def _get_node_import_query(baseEntityLabel: bool, include_source: bool) -> str:
        if baseEntityLabel:
            return (
                f"{INCLUDE_DOCS_QUERY if include_source else ''}"
                "UNWIND $data AS row "
                f"MERGE (source:`{BASE_ENTITY_LABEL}` {{id: row.id}}) "
                "SET source += row.properties "
                "WITH source, row "
                "CALL apoc.create.addLabels(source, [row.type]) YIELD node "
                "RETURN distinct 'done' AS result"
            )
        else:
            return (
                f"{INCLUDE_DOCS_QUERY if include_source else ''}"
                "UNWIND $data AS row "
                "CALL apoc.merge.node([row.type], {id: row.id}, row.properties, {}) YIELD node "
                "RETURN distinct 'done' AS result"
            )
        
    @staticmethod
    def _get_rel_import_query(baseEntityLabel: bool) -> str:
        return (
            "UNWIND $data AS row "
            f"MERGE (source:`{BASE_ENTITY_LABEL}` {{id: row.source}}) "
            f"MERGE (target:`{BASE_ENTITY_LABEL}` {{id: row.target}}) "
            "WITH source, target, row "
            "CALL apoc.merge.relationship(source, row.type, {}, row.properties, target) YIELD rel "
            "RETURN distinct 'done'"
        ) if baseEntityLabel else (
            "UNWIND $data AS row "
            "CALL apoc.merge.node([row.source_label], {id: row.source}, {}, {}) YIELD node as source "
            "CALL apoc.merge.node([row.target_label], {id: row.target}, {}, {}) YIELD node as target "
            "CALL apoc.merge.relationship(source, row.type, {}, row.properties, target) YIELD rel "
            "RETURN distinct 'done'"
        )
    
    @staticmethod
    def _remove_backticks(text: str) -> str:
        return text.replace("`", "")
    
    @staticmethod
    def _dict_to_cypher_map(d: Dict[str, Any]) -> str:
        return "{" + ", ".join([f"{k}: {repr(v)}" for k, v in d.items()]) + "}"

In [3]:
LLMGraphTransformer_default_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "# Knowledge Graph Instructions for GPT-4\n"
                "## 1. Overview\n"
                "You are a top-tier algorithm designed for extracting information in structured "
                "formats to build a knowledge graph. Your goal is to capture entities, relationships, "
                "and associated properties from the input text in a structured manner.\n"
                "- **Nodes** represent entities and concepts.\n"
                "- **Relationships** represent connections between entities and should always include relevant properties and a summarizing sentence.\n"
                "- **Properties** include additional attributes or context for nodes and relationships.\n"
                "## 2. Labeling Nodes\n"
                "- Ensure you use basic or elementary types for node labels. For example, use 'Person' for people.\n"
                "Node IDs should always be names or human-readable identifiers from the text.\n"
                "## 3. Labeling Relationships\n"
                "- Relationships should be labeled with general and timeless relationship types, e.g., 'LOVES', 'FRIEND_OF'.\n"
                "## 4. Coreference Resolution\n"
                "- Maintain entity consistency across references. For example, always refer to 'John Doe' consistently as 'John Doe', even if mentioned as 'Joe' or 'he'.\n"
                "## 5. Strict Compliance\n"
                "Ensure strict adherence to these rules. Non-compliance will result in termination."
            ),
        ),
        (
            "human",
            (
                "Tip: Make sure to answer in the correct format and do not include any explanations.\n"
                "Tip: You can select multiple values for each property if applicable, but you should always choose the most specific properties.\n"
                "Use the given format to extract information from the following input:\n" 
                "{input}.\n"
            ),
        ),
    ]
)

In [4]:
def _get_additional_info(input_type: str) -> str:
    # Check if the input_type is one of the allowed values
    if input_type not in ["node", "relationship", "property"]:
        raise ValueError("input_type must be 'node', 'relationship', or 'property'")

    # Perform actions based on the input_type
    if input_type == "node":
        return (
            "Ensure you use basic or elementary types for node labels.\n"
            "For example, when you identify an entity representing a person, "
            "always label it as **'Person'**. Avoid using more specific terms "
            "like 'Mathematician' or 'Scientist'"
        )
    elif input_type == "relationship":
        return (
            "Instead of using specific and momentary types such as "
            "'BECAME_PROFESSOR', use more general and timeless relationship types "
            "like 'PROFESSOR'. However, do not sacrifice any accuracy for generality"
        )
    elif input_type == "property":
        return ""
    return ""


def optional_enum_field(
    enum_values: Optional[List[str]] = None,
    description: str = "",
    input_type: str = "node",
    llm_type: Optional[str] = None,
    **field_kwargs: Any,
) -> Any:
    """Utility function to conditionally create a field with an enum constraint."""
    
    if enum_values and llm_type == "openai-chat":
        return Field(
            None,  # Allows the field to be optional
            description=f"{description}. Available options are {enum_values}",
            **field_kwargs,
        )
    elif enum_values:
        return Field(
            None,  # Optional field
            description=f"{description}. Available options are {enum_values}",
            **field_kwargs,
        )
    else:
        additional_info = _get_additional_info(input_type)
        return Field(
            None,  # Allows it to be omitted
            description=description + additional_info,
            **field_kwargs,
        )


class _Graph(BaseModel):
    nodes: Optional[List]
    relationships: Optional[List]


class UnstructuredRelation(BaseModel):
    head: str = Field(
        description=(
            "extracted head entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    head_type: str = Field(
        description="type of the extracted head entity like Person, Company, etc"
    )
    relation: str = Field(description="relation between the head and the tail entities")
    tail: str = Field(
        description=(
            "extracted tail entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    tail_type: str = Field(
        description="type of the extracted tail entity like Person, Company, etc"
    )


def create_unstructured_prompt(
    node_labels: Optional[List[str]] = None, rel_types: Optional[List[str]] = None
) -> ChatPromptTemplate:
    node_labels_str = str(node_labels) if node_labels else ""
    rel_types_str = str(rel_types) if rel_types else ""
    base_string_parts = [
        "You are a top-tier algorithm designed for extracting information in "
        "structured formats to build a knowledge graph. Your task is to identify "
        "the entities and relations requested with the user prompt from a given "
        "text. You must generate the output in a JSON format containing a list "
        'with JSON objects. Each object should have the keys: "head", '
        '"head_type", "relation", "tail", and "tail_type". The "head" '
        "key must contain the text of the extracted entity with one of the types "
        "from the provided list in the user prompt.",
        f'The "head_type" key must contain the type of the extracted head entity, '
        f"which must be one of the types from {node_labels_str}."
        if node_labels
        else "",
        f'The "relation" key must contain the type of relation between the "head" '
        f'and the "tail", which must be one of the relations from {rel_types_str}.'
        if rel_types
        else "",
        f'The "tail" key must represent the text of an extracted entity which is '
        f'the tail of the relation, and the "tail_type" key must contain the type '
        f"of the tail entity from {node_labels_str}."
        if node_labels
        else "",
        "Attempt to extract as many entities and relations as you can. Maintain "
        "Entity Consistency: When extracting entities, it's vital to ensure "
        'consistency. If an entity, such as "John Doe", is mentioned multiple '
        "times in the text but is referred to by different names or pronouns "
        '(e.g., "Joe", "he"), always use the most complete identifier for '
        "that entity. The knowledge graph should be coherent and easily "
        "understandable, so maintaining consistency in entity references is "
        "crucial.",
        "IMPORTANT NOTES:\n- Don't add any explanation and text.",
    ]
    system_prompt = "\n".join(filter(None, base_string_parts))

    system_message = SystemMessage(content=system_prompt)
    parser = JsonOutputParser(pydantic_object=UnstructuredRelation)

    human_prompt = PromptTemplate(
        template="""Based on the following example, extract entities and 
relations from the provided text.\n\n
Use the following entity types, don't use other entity that is not defined below:
# ENTITY TYPES:
{node_labels}

Use the following relation types, don't use other relation that is not defined below:
# RELATION TYPES:
{rel_types}

Below are a number of examples of text and their extracted entities and relationships.
{examples}

For the following text, extract entities and relations as in the provided example.
{format_instructions}\nText: {input}""",
        input_variables=["input"],
        partial_variables={
            "format_instructions": parser.get_format_instructions(),
            "node_labels": node_labels,
            "rel_types": rel_types,
            "examples": examples,
        },
    )

    human_message_prompt = HumanMessagePromptTemplate(prompt=human_prompt)

    chat_prompt = ChatPromptTemplate.from_messages(
        [system_message, human_message_prompt]
    )
    return chat_prompt

def create_simple_model(
    node_labels: Optional[List[str]] = None,
    rel_types: Optional[List[str]] = None,
    node_properties: Union[bool, Dict[str, Optional[List[str]]], List[str]] = False,
    llm_type: Optional[str] = None,
    relationship_properties: Optional[List[str]]=None,
    relationship_property_descriptions: Optional[Dict[str, str]] = None,
) -> Type[_Graph]:
    """
    Create a simple graph model with optional constraints on node
    and relationship types.

    Args:
        node_labels (Optional[List[str]]): Specifies the allowed node types.
            Defaults to None, allowing all node types.
        rel_types (Optional[List[str]]): Specifies the allowed relationship types.
            Defaults to None, allowing all relationship types.
        node_properties (Union[bool, List[str], Dict[str, Optional[List[str]]]]): 
            Specifies if node properties should be included. If a list is provided, 
            only properties with keys in the list will be included. If True, all 
            properties are included. If a dictionary is provided, keys are categories 
            and values are lists of properties or None. Defaults to False.
        relationship_properties (Union[bool, Dict[str, Optional[List[str]]], List[str]]): 
            Specifies if relationship properties should be included. If a list is 
            provided, only properties with keys in the list will be included. If True, 
            all properties are included. If a dictionary is provided, keys are 
            categories and values are lists of properties or None. Defaults to False.
        llm_type (Optional[str]): The type of the language model. Defaults to None.
            Only openai supports enum param: openai-chat.
        relationship_property_descriptions (Optional[Dict[str, str]]): Descriptions 
            for each relationship property category.

    Returns:
        Type[_Graph]: A graph model with the specified constraints.

    Raises:
        ValueError: If 'id' is included in the node or relationship properties list.
    """
    
    category_definitions = relationship_property_descriptions or {}

    def process_properties(properties, input_type, min_items=None):
        return optional_enum_field(
                        properties,
                        description="Property key",
                        input_type=input_type,
                        llm_type=llm_type)

    node_fields: Dict[str, Tuple[Any, Any]] = {
        "id": (
            str,
            Field(..., description="Name or human-readable unique identifier."),
        ),
        "status": (
            str,
            "on"
        ),
        "type": (
            str,
            optional_enum_field(
                node_labels,
                description="The type or label of the node.",
                input_type="node",
                llm_type=llm_type,
            ),
        ),
    }
    if node_properties:
        if isinstance(node_properties, list) and "id" in node_properties:
            raise ValueError("The node property 'id' is reserved and cannot be used.")
        
        property_fields = process_properties(node_properties, "property")
        PropertyModel = create_model(
            "Property",
            **{str(field["key"]): (List[str], field["value"]) for field in property_fields}
        )
        node_fields["properties"] = (
            Optional[List[PropertyModel]],
            Field(None, description="List of node properties"),
        )

    SimpleNode = create_model("SimpleNode", **node_fields)  # type: ignore

    relationship_fields: Dict[str, Tuple[Any, Any]] = {
        "source_node_id": (
            str,
            Field(
                ...,
                description="Name or human-readable unique identifier of source node",
            ),
        ),
        "source_node_type": (
            str,
            optional_enum_field(
                node_labels,
                description="The type or label of the source node.",
                input_type="node",
            ),
        ),
        "target_node_id": (
            str,
            Field(
                ...,
                description="Name or human-readable unique identifier of target node",
            ),
        ),
        "target_node_type": (
            str,
            optional_enum_field(
                node_labels,
                description="The type or label of the target node.",
                input_type="node",
            ),
        ),
        "type": (
            str,
            optional_enum_field(
                rel_types,
                description="The type of the relationship.",
                input_type="relationship",
            ),
        ),
        "sentence": (
            str,
            Field(
                default="",
                description="Put the triple source-rel-target into words.",
            ),
        ),
    }
    if relationship_properties:
        if isinstance(relationship_properties, list) and "id" in relationship_properties:
            raise ValueError("The relationship property 'id' is reserved and cannot be used.")

        # Ensure correct type annotation
        relationship_property_field = (List[str], process_properties(relationship_properties, "property"))

        # Create a Pydantic model for relationship properties
        RelationshipPropertyModel = create_model(
            "RelationshipProperty",
            relationship_property=relationship_property_field  # Pass as a tuple with type
        )

        relationship_fields["properties"] = (
            Optional[List[RelationshipPropertyModel]],  # Use list if needed
            Field(None, description="List of relationship properties"),
        )
    SimpleRelationship = create_model("SimpleRelationship", **relationship_fields)  # type: ignore

    class DynamicGraph(_Graph):
        """Represents a graph document consisting of nodes and relationships."""

        nodes: Optional[List[SimpleNode]] = Field(description="List of nodes")  # type: ignore
        relationships: Optional[List[SimpleRelationship]] = Field(  # type: ignore
            description="List of relationships"
        )

    return DynamicGraph

def map_to_base_node(node: Any) -> Node:
    """Map the SimpleNode to the base Node."""
    properties = {}
    if hasattr(node, "properties") and rel.nodes:
        for p in node.properties:
            for key in dict(p):
                if dict(p)[key] != []:
                    properties[format_property_key(key)] = dict(p)[key]
    return Node(id=node.id, type=node.type, properties=properties)

def map_to_base_relationship(rel: Any) -> Relationship:
    """Map the SimpleRelationship to the base Relationship."""
    source = Node(id=rel.source_node_id, type=rel.source_node_type)
    target = Node(id=rel.target_node_id, type=rel.target_node_type)
    properties = {}
    if hasattr(rel, "properties") and rel.properties:
        for p in rel.properties:
            for key, value in dict(p).items():
                if value != None:
                    # Ensure the value is a list
                    if not isinstance(value, list):
                        value = [value]
                    properties[format_property_key(key)] = value
    properties["sentence"]=rel.sentence if rel.sentence else ""
    return Relationship(
        source=source, target=target, type=rel.type, properties=properties
    )


def _parse_and_clean_json(
    argument_json: Dict[str, Any],
) -> Tuple[List[Node], List[Relationship]]:
    nodes = []
    for node in argument_json["nodes"]:
        if not node.get("id"):  # Id is mandatory, skip this node
            continue
        node_properties = {}
        if "properties" in node and node["properties"]:
            for p in node["properties"]:
                node_properties[format_property_key(p["key"])] = p["value"]
        nodes.append(
            Node(
                id=node["id"],
                type=node.get("type"),
                properties=node_properties,
            )
        )
    relationships = []
    for rel in argument_json["relationships"]:
        # Mandatory props
        if (
            not rel.get("source_node_id")
            or not rel.get("target_node_id")
            or not rel.get("type")
        ):
            continue

        # Node type copying if needed from node list
        if not rel.get("source_node_type"):
            try:
                rel["source_node_type"] = [
                    el.get("type")
                    for el in argument_json["nodes"]
                    if el["id"] == rel["source_node_id"]
                ][0]
            except IndexError:
                rel["source_node_type"] = None
        if not rel.get("target_node_type"):
            try:
                rel["target_node_type"] = [
                    el.get("type")
                    for el in argument_json["nodes"]
                    if el["id"] == rel["target_node_id"]
                ][0]
            except IndexError:
                rel["target_node_type"] = None

        rel_properties = {}
        if "properties" in rel and rel["properties"]:
            for p in rel["properties"]:
                rel_properties[format_property_key(p["key"])] = p["value"]

        source_node = Node(
            id=rel["source_node_id"],
            type=rel["source_node_type"],
        )
        target_node = Node(
            id=rel["target_node_id"],
            type=rel["target_node_type"],
        )
        relationships.append(
            Relationship(
                source=source_node,
                target=target_node,
                type=rel["type"],
                properties=rel_properties,
            )
        )
    return nodes, relationships


def _format_nodes(nodes: List[Node]) -> List[Node]:
    return [
        Node(
            id=el.id.title() if isinstance(el.id, str) else el.id,
            type=el.type.capitalize(),
            properties=el.properties,
        )
        for el in nodes
    ]


def _format_relationships(rels: List[Relationship]) -> List[Relationship]:
    return [
        Relationship(
            source=_format_nodes([el.source])[0],
            target=_format_nodes([el.target])[0],
            type=el.type.replace(" ", "_").upper(),
            properties=el.properties,
        )
        for el in rels
    ]


def format_property_key(s: str) -> str:
    words = s.split()
    if not words:
        return s
    first_word = words[0].lower()
    capitalized_words = [word.capitalize() for word in words[1:]]
    return "".join([first_word] + capitalized_words)



def _convert_to_graph_document(
    raw_schema: Dict[Any, Any],
) -> Tuple[List[Node], List[Relationship]]:
    # If there are validation errors
    if not raw_schema["parsed"]:
        try:
            try:  # OpenAI type response
                argument_json = json.loads(
                    raw_schema["raw"].additional_kwargs["tool_calls"][0]["function"][
                        "arguments"
                    ]
                )
            except Exception:  # Google type response
                argument_json = json.loads(
                    raw_schema["raw"].additional_kwargs["function_call"]["arguments"]
                )

            nodes, relationships = _parse_and_clean_json(argument_json)
        except Exception:  # If we can't parse JSON
            return ([], [])
    else:  # If there are no validation errors use parsed pydantic object
        parsed_schema: _Graph = raw_schema["parsed"]
        nodes = (
            [map_to_base_node(node) for node in parsed_schema.nodes]
            if parsed_schema.nodes
            else []
        )

        relationships = (
            [map_to_base_relationship(rel) for rel in parsed_schema.relationships]
            if parsed_schema.relationships
            else []
        )
    # Title / Capitalize
    return _format_nodes(nodes), _format_relationships(relationships)

class LLMGraphTransformer:
    """Transform documents into graph-based documents using a LLM.

    It allows specifying constraints on the types of nodes and relationships to include
    in the output graph. The class supports extracting properties for both nodes and
    relationships.

    Args:
        llm (BaseLanguageModel): An instance of a language model supporting structured
          output.
        allowed_nodes (List[str], optional): Specifies which node types are
          allowed in the graph. Defaults to an empty list, allowing all node types.
        allowed_relationships (List[str], optional): Specifies which relationship types
          are allowed in the graph. Defaults to an empty list, allowing all relationship
          types.
        prompt (Optional[ChatPromptTemplate], optional): The prompt to pass to
          the LLM with additional instructions.
        strict_mode (bool, optional): Determines whether the transformer should apply
          filtering to strictly adhere to `allowed_nodes` and `allowed_relationships`.
          Defaults to True.
        node_properties (Union[bool, List[str]]): If True, the LLM can extract any
          node properties from text. Alternatively, a list of valid properties can
          be provided for the LLM to extract, restricting extraction to those specified.
        relationship_properties (Union[bool, List[str]]): If True, the LLM can extract
          any relationship properties from text. Alternatively, a list of valid
          properties can be provided for the LLM to extract, restricting extraction to
          those specified.

    Example:
        .. code-block:: python
            from langchain_experimental.graph_transformers import LLMGraphTransformer
            from langchain_core.documents import Document
            from langchain_openai import ChatOpenAI

            llm=ChatOpenAI(temperature=0)
            transformer = LLMGraphTransformer(
                llm=llm,
                allowed_nodes=["Person", "Organization"])

            doc = Document(page_content="Elon Musk is suing OpenAI")
            graph_documents = transformer.convert_to_graph_documents([doc])
    """

    def __init__(
        self,
        llm: BaseLanguageModel,
        char_name: str,
        allowed_nodes: List[str] = [],
        allowed_relationships: List[str] = [],
        prompt: Optional[ChatPromptTemplate] = None,
        strict_mode: bool = True,
        node_properties: Union[bool, Dict[str, Optional[List[str]]], List[str]] = False,
        relationship_properties: Union[bool, Dict[str, Optional[List[str]]], List[str]] = False,
        relationship_property_descriptions: Optional[Dict[str, str]] = None,
    ) -> None:
        self.allowed_nodes = allowed_nodes
        self.allowed_relationships = allowed_relationships
        self.strict_mode = strict_mode
        self.relationship_property_descriptions = relationship_property_descriptions
        self.relationship_properties = list(relationship_property_descriptions.keys())
        self._function_call = True
        self.char_name=char_name
        # Check if the LLM really supports structured output
        try:
            llm.with_structured_output(_Graph)
        except NotImplementedError:
            self._function_call = False
        if not self._function_call:
            if node_properties or relationship_properties:
                raise ValueError(
                    "The 'node_properties' and 'relationship_properties' parameters "
                    "cannot be used in combination with a LLM that doesn't support "
                    "native function calling."
                )
            try:
                import json_repair  # type: ignore

                self.json_repair = json_repair
            except ImportError:
                raise ImportError(
                    "Could not import json_repair python package. "
                    "Please install it with `pip install json-repair`."
                )
            prompt = prompt or create_unstructured_prompt(
                allowed_nodes, allowed_relationships
            )
            self.chain = prompt | llm
        else:
            # Define chain
            try:
                llm_type = llm._llm_type  # type: ignore
            except AttributeError:
                llm_type = None
            schema = create_simple_model(
                allowed_nodes,
                allowed_relationships,
                node_properties,
                llm_type,
                relationship_properties,
                relationship_property_descriptions,
            )
            structured_llm = llm.with_structured_output(schema, include_raw=True)
            prompt = prompt or LLMGraphTransformer_default_prompt
            self.chain = prompt | structured_llm


    def process_response(self, document: Document) -> GraphDocument:
        """
        Processes a single document, transforming it into a graph document using
        an LLM based on the model's schema and constraints.
        """
        text = document.page_content
        raw_schema = self.chain.invoke({"input": text, "char_name":self.char_name})
        
        if self._function_call:
            raw_schema = cast(Dict[Any, Any], raw_schema)
            nodes, relationships = _convert_to_graph_document(raw_schema)
        else:
            nodes_set = set()
            relationships = []
            if not isinstance(raw_schema, str):
                raw_schema = raw_schema.content
            parsed_json = self.json_repair.loads(raw_schema)
            for rel in parsed_json:
                # Nodes need to be deduplicated using a set
                nodes_set.add((rel["head"], rel["head_type"]))
                nodes_set.add((rel["tail"], rel["tail_type"]))

                source_node = Node(id=rel["head"], type=rel["head_type"])
                target_node = Node(id=rel["tail"], type=rel["tail_type"])
                rel["properties"]["sentence"] = rel["sentence"]
                relationships.append(
                    Relationship(
                        source=source_node, target=target_node, type=rel["relation"], properties=rel["properties"]
                    )
                )
            # Create nodes list
            nodes = [Node(id=el[0], type=el[1]) for el in list(nodes_set)]

        # Strict mode filtering
        if self.strict_mode and (self.allowed_nodes or self.allowed_relationships):
            if self.allowed_nodes:
                lower_allowed_nodes = [el.lower() for el in self.allowed_nodes]
                nodes = [
                    node for node in nodes if node.type.lower() in lower_allowed_nodes
                ]
                relationships = [
                    rel
                    for rel in relationships
                    if rel.source.type.lower() in lower_allowed_nodes
                    and rel.target.type.lower() in lower_allowed_nodes
                ]
            if self.allowed_relationships:
                relationships = [
                    rel
                    for rel in relationships
                    if rel.type.lower()
                    in [el.lower() for el in self.allowed_relationships]
                ]

        return GraphDocument(nodes=nodes, relationships=relationships, source=Document(page_content = text))


    def convert_to_graph_documents(
        self, documents: Sequence[Document]
    ) -> List[GraphDocument]:
        """Convert a sequence of documents into graph documents.

        Args:
            documents (Sequence[Document]): The original documents.
            **kwargs: Additional keyword arguments.

        Returns:
            Sequence[GraphDocument]: The transformed documents as graphs.
        """
        return [self.process_response(document) for document in documents]


    async def aprocess_response(self, document: Document) -> GraphDocument:
        """
        Asynchronously processes a single document, transforming it into a
        graph document.
        """
        text = document.page_content
        raw_schema = await self.chain.ainvoke({"input": text})
        raw_schema = cast(Dict[Any, Any], raw_schema)
        nodes, relationships = _convert_to_graph_document(raw_schema)

        if self.strict_mode and (self.allowed_nodes or self.allowed_relationships):
            if self.allowed_nodes:
                lower_allowed_nodes = [el.lower() for el in self.allowed_nodes]
                nodes = [
                    node for node in nodes if node.type.lower() in lower_allowed_nodes
                ]
                relationships = [
                    rel
                    for rel in relationships
                    if rel.source.type.lower() in lower_allowed_nodes
                    and rel.target.type.lower() in lower_allowed_nodes
                ]
            if self.allowed_relationships:
                relationships = [
                    rel
                    for rel in relationships
                    if rel.type.lower()
                    in [el.lower() for el in self.allowed_relationships]
                ]

        return GraphDocument(nodes=nodes, relationships=relationships, source=document)
    
    # Function to process and add a document to the graph
    def add_document_to_graph(document, nodes_to_be_added, lock):
        try:
            # Convert document into a graph representation
            graph_document = llm_transformer.convert_to_graph_documents([document])
            # Safely append to the shared list
            with lock:
                nodes_to_be_added.append(graph_document)
        except Exception as e:
            print(f"Error processing document {document.metadata.get('id', 'unknown')}: {e}")


    async def aconvert_to_graph_documents(
        self, documents: Sequence[Document]
    ) -> List[GraphDocument]:
        """
        Asynchronously convert a sequence of documents into graph documents.
        """
        tasks = [
            asyncio.create_task(self.aprocess_response(document))
            for document in documents
        ]
        results = await asyncio.gather(*tasks)
        return results

In [5]:
def load_json(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

def save_json(data, file_path):
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)

def substitute_names_in_dict(data_dict, name_dict):
    for key in data_dict:
        if isinstance(data_dict[key], str):
            data_dict[key] = substitute_names(data_dict[key], name_dict)
        elif isinstance(data_dict[key], dict):
            substitute_names_in_dict(data_dict[key], name_dict)
    return data_dict

In [6]:
from pathlib import Path
import os
import re
from dotenv import load_dotenv

# Load environment variables
load_dotenv(Path("/u/sebono/character_KG/.env"))
# Ensure required environment variables are present
assert 'OPENAI_API_KEY' in os.environ, "OPENAI_API_KEY is missing from environment."
assert 'LANGCHAIN_API_KEY' in os.environ, "LANGCHAIN_API_KEY is missing from environment."
assert 'NEO4J_URI' in os.environ, "NEO4J_URI is missing from environment."
assert 'NEO4J_USERNAME' in os.environ, "NEO4J_USERNAME is missing from environment."
assert 'NEO4J_PASSWORD' in os.environ, "NEO4J_PASSWORD is missing from environment."

# Extract environment variables
#switching to local
url = os.environ['NEO4J_URI_LOCAL']
username = os.environ['NEO4J_USERNAME']
password = os.environ['NEO4J_PASSWORD']

# Display connection details (not recommended for production use)
print("url: ", url, "\nusername: ", username, "\npassword: ", password)

url:  bolt://localhost:7690 
username:  neo4j 
password:  test1234


In [7]:
model_name = "gpt-4o-mini"
name="Jibo"
character_skeleton_path="/u/sebono/character_KG/jibo_dataset/skeleton/"
input_file="/u/sebono/character_KG/jibo_dataset/character/Jibo.csv"
change_names=False

In [8]:
from pathlib import Path
import pandas as pd
import json

relationship_property_descriptions_file = Path(character_skeleton_path) / "relationship_property_descriptions.json"
skeleton_file = Path(character_skeleton_path) / "skeleton.json"
chosen_second_layer_nodes = Path(character_skeleton_path) / "chosen_second_layer_nodes.json"
safe_char_name = re.sub(r'[^\w\s]', '', name).replace(' ', '_').lower()
complete_skeleton = load_json(Path(character_skeleton_path) / "relationship_property_descriptions.json")
data = pd.read_csv(input_file).rename(columns={"dia1":"prompt","dia2":f"{name}"})

In [9]:
complete_skeleton.keys()

dict_keys(['character', 'category', 'religious_beliefs'])

In [11]:
if os.path.exists(chosen_second_layer_nodes):
    complete_skeleton["category"] = pd.DataFrame(complete_skeleton)["category"][chosen_second_layer_nodes].to_dict()
relationship_properties=\
    [f"{key}" for key in complete_skeleton["category"]]+\
    [f"{key}" for key in complete_skeleton["character"]]

relationship_property_descriptions={
    **{f"{key}": value for key, value in complete_skeleton["category"].items()}, 
    **{f"{key}": value for key, value in complete_skeleton["character"].items()},
}

char_name=name
graph_manager = GraphManager(
    uri=url,
    user=username,
    password=password,
    first_node= name,
    relationship_property_descriptions=relationship_property_descriptions
)

100%|██████████████████████████| 10/10 [00:00<00:00, 24.06it/s]


In [12]:
complete_skeleton = load_json(relationship_property_descriptions_file)

# Substitute names in the data
if change_names:
    data["prompt"] = data["prompt"].apply(lambda x: substitute_names(x, name_dict))
    data[char_name] = data[char_name].apply(lambda x: substitute_names(x, name_dict))
data.rename(columns={char_name: name}, inplace=True)

# Create documents after replacing names in the data
docs = [
    Document(
        page_content=f"'prior utterance': {document['prompt']}, '{name}': {document[name]}",
    ) 
    for document in data.to_dict(orient='records')
]

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=24)
documents_to_be_sampled = text_splitter.split_documents(docs)
documents = sample(documents_to_be_sampled, min(100,len(documents_to_be_sampled)))

In [13]:
# Output sample documents
llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")
llm_transformer = LLMGraphTransformer(
    llm=llm,
    char_name=char_name,
    relationship_properties = relationship_properties,
    relationship_property_descriptions = relationship_property_descriptions
)
nodes_to_be_added = []
lock = threading.Lock()

In [14]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from tqdm import tqdm
import logging

# Initialize logger
logger = logging.getLogger(__name__)

# Global variables
num_threads = 32
nodes_to_be_added = []
lock = Lock()

# Function to process and add a document to the graph
def add_document_to_graph(document):
    try:
        # Convert document into a graph representation
        graph_document = llm_transformer.convert_to_graph_documents([document])
        # Safely append to the shared list
        with lock:
            nodes_to_be_added.append(graph_document)
    except Exception as e:
        logger.error(f"Error processing document {document.metadata.get('id', 'unknown')}: {e}")

# Use ThreadPoolExecutor for concurrent document processing
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = [executor.submit(add_document_to_graph, document) for document in documents]
    for future in tqdm(as_completed(futures), total=len(documents), desc="Processing Documents"):
        try:
            future.result()  # Raise any exceptions from the thread
        except Exception as e:
            logger.error(f"Exception in thread: {e}")

# Flatten the nested list of graph documents
flattened_nodes = [node for sublist in nodes_to_be_added for node in sublist]

Processing Documents: 100%|██| 100/100 [00:19<00:00,  5.20it/s]


In [17]:
graph_manager.add_graph_documents(flattened_nodes, include_source=True)

In [18]:
flattened_nodes

[GraphDocument(nodes=[Node(id='Jibo', type='Person')], relationships=[Relationship(source=Node(id='Jibo', type='Person'), target=Node(id='Christmas', type='Event'), type='DETESTS', properties={'relationship_property': ['Jibo_conveys_emotion'], 'sentence': 'Jibo detests Christmas.'})], source=Document(metadata={'id': '64119b6d90171151ad51bc0c9128761b'}, page_content="'prior utterance': i detest christmas, 'Jibo': Well , in not too long it'll be a thing of the past.")),
 GraphDocument(nodes=[Node(id='Jibo', type='Person')], relationships=[Relationship(source=Node(id='Jibo', type='Person'), target=Node(id='Whale', type='Concept'), type='MENTIONS', properties={'sentence': 'Jibo mentions a whale as an example of an NP hard problem.'})], source=Document(metadata={'id': 'ee7e9b6ab5c308e0d865c4ce0e4b6a16'}, page_content="'prior utterance': give me an example of an np hard problem, 'Jibo':  How about for now I just give you this whale.")),
 GraphDocument(nodes=[Node(id='Jibo', type='Robot')], r

In [19]:
# Perform graph maintenance operations
try:
    graph_manager.merge_duplicate_nodes()
    graph_manager.merge_duplicate_relationships()
    graph_manager.reconfigure_all_connections({"relationship_property":relationship_properties}, char_name)
    graph_manager.merge_duplicate_nodes()
    graph_manager.merge_duplicate_relationships()
except Exception as e:
    logger.error(f"Error during graph maintenance: {e}")


            MATCH (source {id: 'Jibo'})-[r]-(target)
            WHERE ANY(prop IN keys(r) WHERE prop = 'relationship_property' AND 'Jibo_perception' IN r[prop])
            WITH source, r, target, type(r) AS relType, properties(r) AS props

            // Create or find the intermediate node
            MERGE (intermediate:Jibo_perception {id: 'Jibo_perception'})
            SET intermediate.type = 'Intermediate_relationship_property', intermediate.subgraphid = 'SecLayerNode', intermediate.description="Subcategories related to Jibo's perception abilities, including emotions, thoughts, wishes, expressions, and wants."
            SET source.subgraphid = 'SecLayerNode'

            // Connect the intermediate node to the subgraph
            WITH source, target, intermediate, props, relType, r
            MATCH (subgraph {id: 'Jibo'})
            WHERE intermediate <> subgraph
            MERGE (subgraph)-[:HAS_SUBGRAPH]-(intermediate)
              ON CREATE SET subgraph.subgraphid = 


            MATCH (source {id: 'Jibo'})-[r]-(target)
            WHERE ANY(prop IN keys(r) WHERE prop = 'relationship_property' AND 'Jibo_obeys_physical_laws' IN r[prop])
            WITH source, r, target, type(r) AS relType, properties(r) AS props

            // Create or find the intermediate node
            MERGE (intermediate:Jibo_obeys_physical_laws {id: 'Jibo_obeys_physical_laws'})
            SET intermediate.type = 'Intermediate_relationship_property', intermediate.subgraphid = 'SecLayerNode', intermediate.description="The character should move according to the physical rules of its world, maintaining internal consistency."
            SET source.subgraphid = 'SecLayerNode'

            // Connect the intermediate node to the subgraph
            WITH source, target, intermediate, props, relType, r
            MATCH (subgraph {id: 'Jibo'})
            WHERE intermediate <> subgraph
            MERGE (subgraph)-[:HAS_SUBGRAPH]-(intermediate)
              ON CREATE SET subgr


            MATCH (source {id: 'Jibo'})-[r]-(target)
            WHERE ANY(prop IN keys(r) WHERE prop = 'relationship_property' AND 'Jibo_conveys_emotion' IN r[prop])
            WITH source, r, target, type(r) AS relType, properties(r) AS props

            // Create or find the intermediate node
            MERGE (intermediate:Jibo_conveys_emotion {id: 'Jibo_conveys_emotion'})
            SET intermediate.type = 'Intermediate_relationship_property', intermediate.subgraphid = 'SecLayerNode', intermediate.description="The character should convey emotions that are appropriate to its circumstances, motivations, and goals, in a clear and legible manner."
            SET source.subgraphid = 'SecLayerNode'

            // Connect the intermediate node to the subgraph
            WITH source, target, intermediate, props, relType, r
            MATCH (subgraph {id: 'Jibo'})
            WHERE intermediate <> subgraph
            MERGE (subgraph)-[:HAS_SUBGRAPH]-(intermediate)
              ON